# CLIP + Logistic Regression (linear probe)
        **vision › classification** · notebook `clip_linear_probe`

        **Kapan dipakai:** Embedding CLIP sangat kuat untuk gambar umum; classifier linear di atasnya sering menyamai fine-tuning dengan data sedikit.

        **Jebakan umum:** Butuh embedding sekali (cepat di GPU). Domain khusus (medis) kurang cocok.

> ⚠️ Notebook ini berat (butuh torch / bobot / GPU). Lihat sel pertama.

**Data demo:** CIFAR-10 subset (4000 latih / 1000 uji, 10 kelas, 32x32). Bobot pretrained dibaca dari `pretrained/` (offline); GPU dipakai otomatis bila ada.

## 0. Setup
Import dan lokasi proyek. Tidak perlu diubah.

In [ ]:
import sys, pathlib, warnings
ROOT = next(p for p in [pathlib.Path.cwd().resolve(), *pathlib.Path.cwd().resolve().parents]
            if (p / "src" / "superdb").is_dir())
sys.path.insert(0, str(ROOT / "src"))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.max_columns", 50)
from superdb.common import *
from superdb.paths import out_dir

## 1. Konfigurasi
Ganti bagian ini saja saat memakai data lomba. Format data: folder-per-kelas atau CSV `path,label`.

In [ ]:
DATA_ROOT  = ROOT / "data_latihan/vision/classification/cifar10_small"            # folder gambar
TRAIN_CSV  = DATA_ROOT / "train.csv"   # kolom: path (relatif ke DATA_ROOT), label
TEST_CSV   = DATA_ROOT / "test.csv"    # kolom: id, path (tanpa label)
TEST_LABELS = DATA_ROOT / "test_labels.csv"  # kunci jawaban test (hanya untuk latihan); None di lomba
SEED, VALID_SIZE = 42, 0.2
MODEL_KEY = "clip_linear_probe"
OUT = out_dir("vision", "classification", MODEL_KEY)
set_seed(SEED)
TEMPLATE = "a photo of a {}"            # prompt untuk zero-shot; coba beberapa variasi

## 2. Baca gambar

In [ ]:
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from superdb.vision import load_images, show_grid
tr = pd.read_csv(TRAIN_CSV); te = pd.read_csv(TEST_CSV)
le = LabelEncoder(); y = le.fit_transform(tr["label"]); CLASSES = list(le.classes_)
X = load_images(tr["path"], DATA_ROOT); Xte = load_images(te["path"], DATA_ROOT)
print(X.shape, Xte.shape, "| kelas:", CLASSES); show_grid(X[::400], [CLASSES[i] for i in y[::400]], n=10); plt.show()
Xtr, Xva, ytr, yva = train_test_split(X, y, test_size=VALID_SIZE, random_state=SEED, stratify=y)

## 3. Embedding gambar dengan CLIP (di-cache oleh pemanggilan berikutnya via disk)

In [ ]:
from pathlib import Path
from superdb.vision import ClipEmbedder
import time; t0 = time.time(); clip = ClipEmbedder()
E, Ete = clip.images(X), clip.images(Xte); print(f"embedding {E.shape} dalam {time.time() - t0:.0f} dtk")

## 4. Linear probe: Logistic Regression di atas embedding CLIP

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, StratifiedKFold
Etr, Eva, ytr_, yva_ = train_test_split(E, y, test_size=VALID_SIZE, random_state=SEED, stratify=y)
model = LogisticRegression(C=10, max_iter=3000)
cv = cross_val_score(model, Etr, ytr_, cv=StratifiedKFold(3, shuffle=True, random_state=SEED), scoring="f1_macro"); print(f"CV f1_macro = {cv.mean():.4f} ± {cv.std():.4f}")
model.fit(Etr, ytr_); pred = model.predict(Eva); proba = model.predict_proba(Eva)
metrics = classification_metrics(yva_, pred, proba, classes=np.arange(len(CLASSES))); display(show_table(metrics))
plot_confusion(yva_, pred, labels=np.arange(len(CLASSES)), path=OUT / "confusion.png"); plt.show()
from sklearn.base import clone
final = clone(model).fit(E, y); CLASSES_PRED = final.predict(Ete)

In [ ]:
pred_te = CLASSES_PRED
sub = pd.DataFrame({"id": te["id"], "label": [CLASSES[i] for i in pred_te]}); sub.to_csv(OUT / "submission.csv", index=False); display(sub.head(3))
if TEST_LABELS is not None and Path(TEST_LABELS).exists():
    truth = pd.read_csv(TEST_LABELS); acc = float((truth.set_index("id").loc[sub["id"], "label"].to_numpy() == sub["label"].to_numpy()).mean())
    print(f"akurasi pada test (kunci jawaban): {acc:.4f}"); metrics["test_accuracy"] = acc

In [ ]:
save_json({"valid_or_train_eval": metrics}, OUT / "metrics.json"); print("OK. hasil di", OUT)

## Langkah berikutnya
- Zero-shot: tulis prompt yang menggambarkan domain ('a satellite photo of ...'), ensemble beberapa template.
- Embedding CLIP juga bagus untuk pencarian gambar mirip dan clustering (lihat `retrieval`).